# 00 — Context

**Context Demystifies Forecasting**

This notebook introduces the core idea behind the repo:

> Forecasting improves when constraints shape hidden state rather than only predictions.

Inspired by **Phys-JEPA: A Physics-informed Joint Embedding Predictive Architecture for Time Series Forecasting**  
arXiv:2606.16076v1


## Lab report framing

Most forecasting models learn a hidden representation of past observations and decode that representation into future predictions.

The key question for this repo:

> Should physical constraints be applied only to final predictions, or should they shape the latent state where the model represents the system?

Phys-JEPA argues for the second option: constrain the state transition inside latent space.


## Three forecasting modes

We will use this repo to compare three simplified modes:

1. **Unconstrained forecasting**  
   The model predicts future values with no physical structure.

2. **Output-constrained forecasting**  
   A correction is applied after prediction.

3. **Latent-constrained forecasting**  
   The hidden state itself is decomposed and constrained before prediction.

The repo's central experiment is:

> Do latent constraints outperform output constraints?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"RESULTS: {RESULTS}")

## Toy system

Before working with real climate, traffic, or electricity data, we begin with a simple periodic process.

This is not a reproduction of Phys-JEPA. It is a small teaching model for the repo's core idea:

> a system has observable values, hidden state, and transition structure.


In [ ]:
t = np.linspace(0, 12 * np.pi, 600)

signal = np.sin(t)
trend = 0.015 * t
noise = 0.08 * np.random.default_rng(260616076).normal(size=len(t))

observed = signal + trend + noise

df = pd.DataFrame({
    "t": t,
    "signal": signal,
    "trend": trend,
    "observed": observed,
})

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["t"], df["observed"], label="observed")
ax.plot(df["t"], df["signal"] + df["trend"], label="structured process")

ax.set_title("Toy forecasting context")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "00_toy_forecasting_context.png", dpi=160)
plt.show()

## Physical state + residual state

Phys-JEPA separates latent state into:

```text
latent_state = physical_state + residual_state
```

For this toy system:

- the **physical state** is the smooth, structured process
- the **residual state** is short-scale variation not explained by that structure

The point is not that all residuals are noise. The point is that latent space can become more interpretable when measurable structure is separated from everything else.


In [ ]:
df["physical_state"] = df["signal"] + df["trend"]
df["residual_state"] = df["observed"] - df["physical_state"]

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["t"], df["physical_state"], label="physical state")
ax.plot(df["t"], df["residual_state"], label="residual state")

ax.set_title("Decomposed toy latent state")
ax.set_xlabel("time")
ax.set_ylabel("component value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "00_physical_residual_context.png", dpi=160)
plt.show()

## Output correction vs latent constraint

A useful distinction:

```text
Output correction:
history → hidden state → prediction → correction

Latent constraint:
history → physical state + residual state → constrained transition → prediction
```

The output correction model waits until after prediction.

The latent constraint model changes the representation before prediction.


In [ ]:
comparison = pd.DataFrame({
    "mode": [
        "unconstrained",
        "output-constrained",
        "latent-constrained",
    ],
    "where_constraint_acts": [
        "nowhere",
        "after prediction",
        "inside hidden state",
    ],
    "repo_question": [
        "How much drift appears?",
        "Can corrections repair drift?",
        "Does state structure reduce drift before decoding?",
    ],
})

comparison

## Notebook map

Planned progression:

| Notebook | Goal |
|---|---|
| `00_context` | Define the lab report question |
| `07_output_vs_latent_constraints` | Compare correction after prediction with constraint inside state |
| `13_physical_residual_decomposition` | Build the central decomposition |
| `17_climate_forecasting` | Try a real climate forecasting dataset |
| `23_constraint_ablations` | Remove components and compare |
| `29_interpretability` | Visualize latent trajectories |
| `37_forecast_stability` | Study long-horizon error accumulation |
| `43_context_demystifies_forecasting` | Synthesize findings |


## Engineering statement

Forecasting improves when constraints shape hidden state rather than only predictions.

When measurable structure organizes latent transitions, models become more interpretable, more stable across time horizons, and better aligned with the systems they represent.

```text
1 + 1 > |1.4i| ≠ $0.96
c = √(E/m)
```
